In [1]:
import sys
dice_path = "/Users/volk/Documents/bau24-25/thesis/repos/DiCE-X"
sys.path.insert(0, dice_path)

In [2]:
%load_ext autoreload
%autoreload 2

In [6]:
from dice_ml_x.utils import helpers
import pandas as pd
import numpy as np
from IPython.display import display

In [7]:
adult_income = helpers.load_adult_income_dataset()
german_credit = helpers.load_german_credit_dataset()
lending_club = helpers.load_lending_club_dataset()

In [22]:
def summarize_dataset(df: pd.DataFrame,
                      continuous_features: list = None,
                      outcome_name: str = None,
                      dataset_name: str = None,
                      top_n_categories: int = 5):
    """
    Return a dataset-level summary and a per-feature summary DataFrame.
    - df: input DataFrame (raw, with outcome column if present)
    - continuous_features: list of continuous feature names (optional)
    - outcome_name: name of target/outcome column (optional)
    - dataset_name: optional label for the dataset
    - top_n_categories: how many top categories to show for categorical features
    Returns: (dataset_summary_df, features_summary_df, target_distribution_df or None)
    """
    # Basic counts
    n_samples = df.shape[0]
    n_features = df.shape[1] - (1 if outcome_name and outcome_name in df.columns else 0)

    # Determine continuous/categorical sets
    if continuous_features is not None:
        cont_set = set(continuous_features)
        cont_cols = [c for c in df.columns if c in cont_set]
        cat_cols = [c for c in df.columns if c not in cont_set and (outcome_name is None or c != outcome_name)]
    else:
        # infer by dtype: numeric -> continuous, object/category/bool -> categorical
        cont_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        # remove outcome if present
        if outcome_name in cont_cols:
            cont_cols.remove(outcome_name)
        cat_cols = [c for c in df.columns if c not in cont_cols and c != outcome_name]

    n_continuous = len(cont_cols)
    n_categorical = len(cat_cols)

    # Memory
    try:
        mem = df.memory_usage(deep=True).sum()
    except Exception:
        mem = df.memory_usage().sum()

    # Missing
    total_missing = df.isna().sum().sum()
    total_cells = df.size
    pct_missing = 100.0 * total_missing / total_cells if total_cells else 0.0

    # Dataset-level summary row
    dataset_summary = {
        'dataset_name': dataset_name or '',
        'n_samples': n_samples,
        'n_features': n_features,
        'n_continuous': n_continuous,
        'n_categorical': n_categorical,
        'total_missing_cells': int(total_missing),
        'pct_missing': round(pct_missing, 3),
        'memory_bytes': int(mem)
    }
    dataset_summary_df = pd.DataFrame([dataset_summary]).set_index('dataset_name')

    # Per-feature summary
    rows = []
    for col in df.columns:
        if outcome_name is not None and col == outcome_name:
            role = 'outcome'
        elif col in cont_cols:
            role = 'continuous'
        else:
            role = 'categorical'

        series = df[col]
        n_missing = int(series.isna().sum())
        pct_missing_col = 100.0 * n_missing / n_samples if n_samples else 0.0
        n_unique = int(series.nunique(dropna=True))
        unique_pct = 100.0 * n_unique / n_samples if n_samples else 0.0
        dtype = str(series.dtype)

        row = {
            'feature': col,
            'role': role,
            'dtype': dtype,
            'n_missing': n_missing,
            'pct_missing': round(pct_missing_col, 3),
            'n_unique': n_unique,
            'unique_pct': round(unique_pct, 3)
        }

        if role == 'continuous':
            stats = series.dropna().describe()
            row.update({
                'min': float(stats.get('min', np.nan)) if not pd.isna(stats.get('min')) else np.nan,
                'q1': float(series.dropna().quantile(0.25)) if n_unique>0 else np.nan,
                'median': float(stats.get('50%', np.nan)) if not pd.isna(stats.get('50%')) else np.nan,
                'q3': float(series.dropna().quantile(0.75)) if n_unique>0 else np.nan,
                'max': float(stats.get('max', np.nan)) if not pd.isna(stats.get('max')) else np.nan,
                'mean': float(stats.get('mean', np.nan)) if not pd.isna(stats.get('mean')) else np.nan,
                'std': float(stats.get('std', np.nan)) if not pd.isna(stats.get('std')) else np.nan,
                'skew': float(series.dropna().skew()) if n_unique>0 else np.nan,
                'top_values': ''
            })
        else:
            # categorical: show top categories
            top = series.dropna().value_counts().head(top_n_categories)
            top_str = '; '.join([f"{val} ({cnt})" for val, cnt in top.items()])
            row.update({
                'min': np.nan, 'q1': np.nan, 'median': np.nan, 'q3': np.nan, 'max': np.nan,
                'mean': np.nan, 'std': np.nan, 'skew': np.nan,
                'top_values': top_str
            })

        rows.append(row)

    features_summary_df = pd.DataFrame(rows).set_index('feature')

    # Target distribution if requested
    target_distribution_df = None
    if outcome_name is not None and outcome_name in df.columns:
        target = df[outcome_name]
        cnts = target.value_counts(dropna=False)
        pct = 100.0 * cnts / len(target)
        target_distribution_df = pd.DataFrame({
            'count': cnts,
            'pct': pct.round(3)
        })

    return dataset_summary_df, features_summary_df, target_distribution_df


# --- Example usage ---
# from dice_ml_x.utils import helpers
# adult_income = helpers.load_adult_income_dataset()
# ds_summary, feat_summary, target_dist = summarize_dataset(adult_income,
#                                                           continuous_features=['age', 'hours_per_week'],
#                                                           outcome_name='income',
#                                                           dataset_name='adult_income')
# display(ds_summary)
# display(feat_summary.head(40))   # display first 40 feature rows
# if target_dist is not None:
#     display(target_dist)
#
# # Save to CSV:
# ds_summary.to_csv("dataset_summary.csv")
# feat_summary.to_csv("features_summary.csv")
# if target_dist is not None:
#     target_dist.to_csv("target_distribution.csv")

In [26]:
datasets = [
    {
        "ds_name": "adult_income",
        "dataset": adult_income,
        "target": "income",
        "continuous_features": ["age", "hours_per_week"]
    },
    {
        "ds_name": "german_credit",
        "dataset": german_credit,
        "target": "credit_risk",
        "continuous_features": list(german_credit.select_dtypes(include=[np.number]).columns.difference(['credit_risk']))
    },
    {
        "ds_name": "lending_club",
        "dataset": lending_club,
        "target": "loan_status",
        "continuous_features": list(lending_club.select_dtypes(include=[np.number]).columns.difference(["loan_status"]))
    }
]

tables = []
for ds_metadata in datasets:
    ds = ds_metadata["dataset"]
    target = ds_metadata["target"]
    cont_features = ds_metadata["continuous_features"]
    ds_name = ds_metadata["ds_name"]
    ds_summary, feat_summary, target_dist = summarize_dataset(ds,
                                                              continuous_features=cont_features,
                                                              outcome_name=target,
                                                              dataset_name=ds_name)
    tables.append(ds_summary)

display(pd.concat(tables))



,n_samples,n_features,n_continuous,n_categorical,total_missing_cells,pct_missing,memory_bytes
dataset_name,,,,,,,
adult_income,32561,8,2,6,0,0.0,13330784
german_credit,1000,20,7,13,0,0.0,1023349
lending_club,39715,8,4,4,0,0.0,11541595


In [31]:
loan_status = pd.read_csv("loan_status.csv")
loan_status = loan_status.drop(columns=["Loan_ID"])
loan_cont_features = loan_status.select_dtypes(include=[np.number]).columns.difference(["Loan_Status"]).tolist()
ds_summary, feat_summary, target_dist = summarize_dataset(loan_status,
                                                              continuous_features=loan_cont_features,
                                                              outcome_name="Loan_Status",
                                                              dataset_name="Loan_Status")
display(ds_summary)

,n_samples,n_features,n_continuous,n_categorical,total_missing_cells,pct_missing,memory_bytes
dataset_name,,,,,,,
Loan_Status,614,11,5,6,149,2.022,284307
